# 04 Feature Diagnostics

Vor jeglicher Feature-Konstruktion werden in diesem Notebook numerische Belege
für die Trennschärfe und Relevanz aller 21 Ausgangs-Features erhoben.
Das ist Sprint 1 der Feature-Engineering-Roadmap: **Diagnose & EDA**.

Ziel ist eine begründete **Shortlist von 15–18 Features** für Notebook 05,
die auf vier unabhängigen Metriken (IV, MI, KS, Spearman) beruht.
Zusätzlich wird die Multikollinearitätsstruktur durch eine Spearman-Heatmap
und Variance Inflation Factors (VIF) offengelegt — relevant für Modellwahl
und Interpretation in Notebook 07.

**Wichtig:** Das Test-Set wird in diesem Notebook an keiner Stelle berührt.
Alle Analysen basieren ausschließlich auf `X_train` / `y_train`.
Kein Resampling, keine Feature-Konstruktion.

**Struktur:**
1. Setup
2. Datensatz laden und Train/Test-Split rekonstruieren
3. Information Value (IV) via optbinning
4. Mutual Information
5. Kolmogorov-Smirnov und Spearman
6. Konsolidiertes Ranking
7. Multikollinearitätsanalyse
8. Shortlist erstellen und exportieren
9. Zusammenfassung

## 1. Setup

Alle Bibliotheken werden zentral importiert. `optbinning` wird bei Bedarf automatisch
nachinstalliert — dasselbe Muster wie in Notebook 03 für `ucimlrepo`.
Der Seed wird einmalig auf `42` gesetzt und danach konsistent in allen
randomisierten Funktionsaufrufen verwendet.

In [ ]:
import importlib, subprocess, sys

for pkg in ['ucimlrepo', 'optbinning']:
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from optbinning import BinningProcess
from ucimlrepo import fetch_ucirepo

SEED = 42
print('Alle Importe erfolgreich.')

## 2. Datensatz laden und Train/Test-Split rekonstruieren

Der identische stratifizierte 80/20 Split aus Notebook 03 wird hier neu erzeugt
(gleicher `SEED`, gleiche `stratify`-Option). Dadurch ist dieses Notebook
vollständig eigenständig und benötigt keine gespeicherten Parquet-Dateien als
Voraussetzung — die Reproduzierbarkeit ist durch deterministischen Code garantiert,
nicht durch Abhängigkeit von Zwischenständen.

Anschließend werden die Feature-Typ-Gruppen aus Notebook 03 analog neu definiert.
Ein kurzer Shape-Check verifiziert, dass der Split identisch ist.

In [ ]:
dataset = fetch_ucirepo(id=891)
X = dataset.data.features.copy()
y = dataset.data.targets.squeeze().copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

# Feature-Typ-Gruppen (identisch zu Notebook 03)
BINARY_COLS = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex',
]
ORDINAL_COLS = ['GenHlth', 'Age', 'Education', 'Income']
COUNT_COLS   = ['MentHlth', 'PhysHlth']
NUMERIC_COLS = ['BMI']
ALL_FEATURES = BINARY_COLS + ORDINAL_COLS + COUNT_COLS + NUMERIC_COLS

# Shape-Verifikation
assert X_train.shape == (202944, 21), f'Unerwartetes Train-Shape: {X_train.shape}'
assert X_test.shape  == (50736,  21), f'Unerwartetes Test-Shape:  {X_test.shape}'
print(f'Train: {X_train.shape[0]:>7,} Samples, {X_train.shape[1]} Features')
print(f'Test:  {X_test.shape[0]:>7,} Samples  (wird nicht mehr angefasst)')
print(f'Prävalenz Train: {y_train.mean():.4f}  |  Test: {y_test.mean():.4f}')

## 3. Information Value (IV) via optbinning

Der **Information Value** (IV) quantifiziert, wie gut ein Feature die Zielvariable
trennt — nach *optimaler* Binnung der Feature-Werte. Optimale Binnung bedeutet,
dass `BinningProcess` die Schnittgrenzen so wählt, dass der IV maximiert wird,
anstatt gleichbreite oder gleichbesetzte Bins zu erzwingen. Das macht IV besonders
sensitiv gegenüber nichtlinearen, aber monotonen Zusammenhängen.

Berechnung: IV = Σ (WoE_i × (Ereignis_i% − Nicht-Ereignis_i%)) über alle Bins i,
wobei WoE_i = ln(Ereignis_i% / Nicht-Ereignis_i%) (Weight of Evidence).

**Interpretation nach Siddiqi (2006), *Credit Risk Scorecard*:**

| IV-Bereich | Bewertung |
|---|---|
| < 0.02 | unbrauchbar |
| 0.02 – 0.10 | schwach |
| 0.10 – 0.30 | mittel |
| 0.30 – 0.50 | stark |
| > 0.50 | verdächtig (ggf. Datenleck) |

Für binäre Features (0/1) mit sehr schiefem Muster (z. B. CholCheck ~96 % positiv)
kann optbinning nur einen informativen Bin erzeugen — das Ergebnis ist dann
methodisch korrekt ein sehr niedriger IV.

In [ ]:
# BinningProcess: binäre und ordinale Spalten werden als kategorisch übergeben,
# damit optbinning keine künstlichen Schnittgrenzen in Integer-Codes erzeugt
categorical_vars = BINARY_COLS + ORDINAL_COLS

bp = BinningProcess(
    variable_names=ALL_FEATURES,
    categorical_variables=categorical_vars,
)
bp.fit(X_train[ALL_FEATURES], y_train)

bp_summary = bp.summary()
iv_series = bp_summary.set_index('name')['iv'].reindex(ALL_FEATURES)

# Interpretationsstufe zuweisen
def iv_label(v):
    if v < 0.02:  return 'unbrauchbar'
    if v < 0.10:  return 'schwach'
    if v < 0.30:  return 'mittel'
    if v < 0.50:  return 'stark'
    return 'verdächtig'

iv_df = (
    iv_series
    .sort_values(ascending=False)
    .to_frame(name='IV')
    .assign(Bewertung=lambda d: d['IV'].map(iv_label))
)
print(iv_df.to_string(float_format='{:.4f}'.format))

In [ ]:
COLOR_MAP = {
    'unbrauchbar': '#d62728',
    'schwach':     '#ff7f0e',
    'mittel':      '#2ca02c',
    'stark':       '#1f77b4',
    'verdächtig':  '#9467bd',
}

fig, ax = plt.subplots(figsize=(10, 6))
colors = iv_df['Bewertung'].map(COLOR_MAP)
ax.barh(iv_df.index[::-1], iv_df['IV'][::-1], color=colors[::-1], edgecolor='white')

for thresh, lbl in [(0.02, 'schwach'), (0.10, 'mittel'), (0.30, 'stark'), (0.50, 'verdächtig')]:
    ax.axvline(thresh, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.text(thresh + 0.002, 0.5, lbl, transform=ax.get_xaxis_transform(),
            fontsize=7.5, color='grey', va='bottom')

ax.set_xlabel('Information Value (IV)')
ax.set_title('Information Value pro Feature (optimale Binnung via optbinning)')
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

## 4. Mutual Information

**Mutual Information** (MI) misst, wie viel Information ein Feature über die
Zielvariable enthält — ohne jede Annahme über funktionale Form oder Linearität.
MI erfasst damit auch nichtlineare Zusammenhänge, die IV und Spearman
möglicherweise unterschätzen.

Technisch wichtig: `mutual_info_classif` schätzt MI über k-NN-Abstände.
Dabei muss die `discrete_features`-Maske korrekt gesetzt werden:
Für kontinuierliche Features (hier: BMI) nutzt sklearn einen anderen
Schätzer als für diskrete/kategorische Features. Wird BMI fälschlicherweise
als diskret markiert, wird MI unterschätzt; werden binäre Features als
kontinuierlich behandelt, wird MI überschätzt.

Daher: `discrete_features=True` für alle Features **außer** BMI.

In [ ]:
# discrete_features-Maske: True für alle nicht-kontinuierlichen Features
discrete_mask = np.array([col != 'BMI' for col in ALL_FEATURES])

mi_values = mutual_info_classif(
    X_train[ALL_FEATURES],
    y_train,
    discrete_features=discrete_mask,
    random_state=SEED,
)

mi_series = pd.Series(mi_values, index=ALL_FEATURES, name='MI').sort_values(ascending=False)
print(mi_series.to_string(float_format='{:.4f}'.format))

## 5. Kolmogorov-Smirnov und Spearman

**Kolmogorov-Smirnov-Statistik (KS):** Der KS-Test vergleicht die empirischen
kumulativen Verteilungsfunktionen der Feature-Werte in der positiven und negativen
Klasse. Die KS-Statistik ist der maximale vertikale Abstand zwischen beiden
Kurven — ein hoher Wert bedeutet, dass sich die Feature-Verteilungen in den
Klassen stark unterscheiden.
Im Kreditrisiko-Scoring ist KS > 0.20 ein gängiger Mindest-Schwellenwert für
diskriminante Features (Siddiqi 2006). Hier verwenden wir 0.05 als
konservativen Mindest-Schwellenwert für die Shortlist, da viele binäre
Features strukturell begrenzte KS-Werte erzeugen können.

**Spearman-Rangkorrelation:** Misst monotone (nicht notwendig lineare)
Zusammenhänge zwischen Feature und Target. Robuster gegenüber Ausreißern
als Pearson; passend für ordinale und count-basierte Features.
Wir verwenden den Absolutwert, da negative Korrelation (z. B. Einkommen
und Diabetes) genauso informativ ist wie positive.

In [ ]:
y_arr = y_train.values
pos_mask = y_arr == 1
neg_mask = y_arr == 0

ks_values      = {}
spearman_values = {}

for col in ALL_FEATURES:
    feat = X_train[col].values
    ks_stat, _ = stats.ks_2samp(feat[pos_mask], feat[neg_mask])
    sp_corr, _ = stats.spearmanr(feat, y_arr)
    ks_values[col]       = ks_stat
    spearman_values[col] = sp_corr

ks_series = pd.Series(ks_values, name='KS').sort_values(ascending=False)
sp_series = pd.Series(spearman_values, name='Spearman')

print('KS-Statistik (absteigend):')
print(ks_series.to_string(float_format='{:.4f}'.format))
print()
print('Spearman-Korrelation zum Target (absteigend nach |r|):')
print(sp_series.reindex(sp_series.abs().sort_values(ascending=False).index)
              .to_string(float_format='{:.4f}'.format))

## 6. Konsolidiertes Ranking

Die vier Metriken — IV, MI, KS und |Spearman| — werden zu einer einzigen
Übersichtstabelle zusammengeführt. Keine Aggregation zu einem einzigen Score,
da die Metriken unterschiedliche Aspekte messen und ein Kompromiss-Score
Informationen vernichtet.

**Aufnahme in die Shortlist:** Ein Feature wird in die Shortlist aufgenommen,
wenn **mindestens eine** der folgenden Bedingungen erfüllt ist:

- IV > 0.02 (oberste Siddiqi-Grenze für "schwach"; alles darunter ist unbrauchbar)
- MI > 0.005 (niedriger Absolutwert, aber Einheit ist Nats — auch kleine Werte
  können klinisch bedeutsam sein)
- KS > 0.05 (minimale Trennung der Klassenverteilungen)
- |Spearman| > 0.05 (schwache, aber konsistente monotone Assoziation)

Die Oder-Verknüpfung ist bewusst inklusiv: Ein Feature, das nur nach einer
Metrik schwach erscheint, kann nach einer anderen informativer sein
(z. B. nichtlinearer Zusammenhang bei niedrigem Spearman, aber hohem MI).
Ausschluss erfolgt erst, wenn **alle vier Metriken** unterhalb ihrer Schwelle liegen.

In [ ]:
ranking = pd.DataFrame({
    'IV':           iv_series.reindex(ALL_FEATURES),
    'MI':           mi_series.reindex(ALL_FEATURES),
    'KS':           ks_series.reindex(ALL_FEATURES),
    'Spearman_abs': sp_series.abs().reindex(ALL_FEATURES),
}).round(4)

# Shortlist-Regel: mindestens eine Metrik über Schwellenwert
ranking['in_shortlist'] = (
    (ranking['IV'] > 0.02) |
    (ranking['MI'] > 0.005) |
    (ranking['KS'] > 0.05) |
    (ranking['Spearman_abs'] > 0.05)
)

ranking = ranking.sort_values('IV', ascending=False)

# Formatierte Ausgabe
pd.set_option('display.float_format', '{:.4f}'.format)
display(ranking.style
    .background_gradient(subset=['IV', 'MI', 'KS', 'Spearman_abs'], cmap='YlGn')
    .applymap(lambda v: 'background-color: #d4edda' if v else 'background-color: #f8d7da',
              subset=['in_shortlist'])
    .set_caption('Feature-Ranking: IV, MI, KS, |Spearman| — grün = in Shortlist')
)

## 7. Multikollinearitätsanalyse

Für die Modellwahl in Notebook 07 ist die Korrelationsstruktur zwischen Features
genauso entscheidend wie deren individuelle Trennschärfe.

- **Baumbasierte Modelle** (Random Forest, Gradient Boosting) sind gegenüber
  Multikollinearität robust, was Vorhersagen betrifft. Jedoch werden
  **Permutation Importances** und Shapley-Werte verzerrt, wenn Features
  hochkorreliert sind — das Modell kann beliebig zwischen korrelierten Features
  wechseln, ohne die Performance zu verändern.
- **Lineare Modelle** (Logistic Regression) leiden direkt unter
  Multikollinearität: Koeffizienten werden instabil, Standardfehler überhöht,
  und Vorhersagen können trotzdem stabil bleiben.

Die Analyse umfasst drei Teile:
1. Spearman-Korrelationsmatrix (Heatmap) aller 21 Features
2. Variance Inflation Factor (VIF) pro Feature
3. Qualitative Diskussion erwarteter Cluster

### 7.1 Spearman-Korrelationsmatrix

Spearman statt Pearson, weil die meisten Features ordinal oder binär sind.
Pearson würde bei ordinalen und binären Daten die Korrelationsstärke systematisch
unterschätzen. Spearman misst monotone Rangkorrelation — invariant gegenüber
monotonen Transformationen der Skala.

In [ ]:
spearman_matrix = X_train[ALL_FEATURES].apply(
    lambda col: pd.Series(
        [stats.spearmanr(col, X_train[c]).correlation for c in ALL_FEATURES],
        index=ALL_FEATURES,
    )
)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    spearman_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    annot_kws={'size': 7},
    ax=ax,
)
ax.set_title('Spearman-Korrelationsmatrix — alle 21 Features (X_train)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

# Heatmap für Export sichern
out_dir = Path('../outputs/04_diagnostics')
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
print(f'Heatmap gespeichert: {(out_dir / "correlation_heatmap.png").resolve()}')
plt.show()

### 7.2 Variance Inflation Factor (VIF)

Der VIF misst, um welchen Faktor die Varianz eines Koeffizienten durch
Korrelationen mit anderen Features erhöht wird. Berechnung: VIF_j = 1 / (1 − R²_j),
wobei R²_j der Determinationskoeffizient aus der Regression von Feature j
auf alle anderen Features ist.

**Schwellenwerte** (Fahrmeir et al., *Regression*, 2013):
- VIF < 5: unkritisch
- VIF 5–10: problematisch, Modellinterpretation eingeschränkt
- VIF > 10: starke Multikollinearität, Koeffizienten instabil

VIF ist skalenunabhängig, aber numerisch stabiler nach Standardisierung.
Daher wird `StandardScaler` vor der Berechnung angewendet.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train[ALL_FEATURES])

vif_values = [
    variance_inflation_factor(X_scaled, i)
    for i in range(X_scaled.shape[1])
]

vif_df = (
    pd.DataFrame({'Feature': ALL_FEATURES, 'VIF': vif_values})
    .set_index('Feature')
    .sort_values('VIF', ascending=False)
    .round(2)
)

def vif_label(v):
    if v < 5:   return 'unkritisch'
    if v < 10:  return 'problematisch'
    return 'stark multikollinear'

vif_df['Bewertung'] = vif_df['VIF'].map(vif_label)
print(vif_df.to_string())

### 7.3 Erwartete Korrelationscluster

Aus klinischer und sozioökonomischer Logik lassen sich vier Cluster a priori ableiten;
die Heatmap zeigt, ob sie sich empirisch bestätigen:

**1. Kardiometabolischer Cluster** (HighBP, HighChol, HeartDiseaseorAttack, Stroke, BMI):
Diese Risikofaktoren sind über gemeinsame pathophysiologische Pfade verknüpft
(Insulinresistenz, Dyslipidämie, Arteriosklerose). Moderate positive Korrelationen
erwartet. HighBP und HeartDiseaseorAttack dürften dabei am stärksten korrelieren.

**2. Gesundheits-/Mobilitätscluster** (GenHlth, PhysHlth, DiffWalk, MentHlth):
Allgemeiner Gesundheitszustand, körperliche Einschränkungen und Mobilitätsprobleme
verstärken sich gegenseitig. GenHlth als übergeordneter Indikator sollte die
höchsten Korrelationen in dieser Gruppe aufweisen.

**3. Sozioökonomischer Cluster** (Education, Income):
Bildung und Einkommen sind klassisch korreliert (Human Capital Theory).
Da im BRFSS-Datensatz beide als Ordinalskalen vorliegen, sollte dieser
Zusammenhang klar sichtbar sein.

**4. Lifestyle-Cluster** (PhysActivity, Fruits, Veggies, Smoker, HvyAlcoholConsump):
Verhaltenskorrelationen sind typischerweise schwächer als klinische,
da Lebensstilentscheidungen weniger deterministisch zusammenhängen.
HvyAlcoholConsump zeigt im BRFSS häufig negative Korrelation mit Diabetes
(Selection Bias: Heavy Drinker sterben früher oder erscheinen seltener im Panel),
was die Gruppenstruktur aufbrechen kann.

Features, die mehreren Clustern angehören (z. B. DiffWalk tritt sowohl im
Gesundheits- als auch im kardiometabolischen Cluster auf), sind besonders
anfällig für hohe VIF-Werte.

## 8. Shortlist erstellen und exportieren

Die Shortlist wird direkt aus der `in_shortlist`-Spalte des Ranking-DataFrames
extrahiert. Für ausgeschlossene Features wird die Begründung explizit ausgegeben
(welche Schwellen nicht erreicht wurden), um die Entscheidung nachvollziehbar
zu halten.

Drei Artefakte werden in `outputs/04_diagnostics/` gespeichert:
- `feature_diagnostics.csv` — vollständige Ranking-Tabelle inkl. `in_shortlist`
- `correlation_heatmap.png` — bereits in 7.1 gespeichert
- `shortlist.txt` — Shortlist als reine Feature-Namen, plus Kommentarblock
  mit ausgeschlossenen Features und Begründung

In [ ]:
shortlist_features = ranking[ranking['in_shortlist']].index.tolist()
excluded_features  = ranking[~ranking['in_shortlist']].index.tolist()

print(f'Shortlist ({len(shortlist_features)} Features):')
for f in shortlist_features:
    print(f'  {f}')

print(f'\nAusgeschlossen ({len(excluded_features)} Features):')
for f in excluded_features:
    row = ranking.loc[f]
    gründe = []
    if row['IV']           <= 0.02:  gründe.append(f'IV={row["IV"]:.4f}<=0.02')
    if row['MI']           <= 0.005: gründe.append(f'MI={row["MI"]:.4f}<=0.005')
    if row['KS']           <= 0.05:  gründe.append(f'KS={row["KS"]:.4f}<=0.05')
    if row['Spearman_abs'] <= 0.05:  gründe.append(f'|Sp|={row["Spearman_abs"]:.4f}<=0.05')
    print(f'  {f:<25}  ({", ".join(gründe)})')

In [ ]:
out_dir = Path('../outputs/04_diagnostics')
out_dir.mkdir(parents=True, exist_ok=True)

# feature_diagnostics.csv
ranking.to_csv(out_dir / 'feature_diagnostics.csv')
print(f'Gespeichert: feature_diagnostics.csv ({len(ranking)} Zeilen)')

# shortlist.txt
shortlist_lines = ['# Feature Shortlist — 04_feature_diagnostics.ipynb',
                   f'# Shortlist-Regel: IV>0.02 OR MI>0.005 OR KS>0.05 OR |Spearman|>0.05',
                   f'# {len(shortlist_features)} Features in Shortlist, {len(excluded_features)} ausgeschlossen',
                   '']

shortlist_lines += shortlist_features
shortlist_lines += ['', '# Ausgeschlossene Features (alle vier Schwellen unterschritten):']

for f in excluded_features:
    row = ranking.loc[f]
    shortlist_lines.append(
        f'# {f}: IV={row["IV"]:.4f}, MI={row["MI"]:.4f}, '
        f'KS={row["KS"]:.4f}, |Sp|={row["Spearman_abs"]:.4f}'
    )

(out_dir / 'shortlist.txt').write_text('\n'.join(shortlist_lines), encoding='utf-8')
print(f'Gespeichert: shortlist.txt')
print(f'Heatmap bereits gespeichert: correlation_heatmap.png')
print(f'\nAlle Artefakte unter: {out_dir.resolve()}')

## 9. Zusammenfassung

### Wichtigste Erkenntnisse

**1. Top-5 Features nach IV:**
Die Tabelle in Abschnitt 6 zeigt die nach IV absteigend sortierten Features.
Erwartet führen Indikatoren des allgemeinen Gesundheitszustands (GenHlth, BMI,
HighBP) zusammen mit sozioökonomischen Faktoren (Age, Income) das Ranking an.
Diese decken sich mit dem Literaturstand zu BRFSS-basierten Diabetes-Modellen
(Xie & Huang 2019; Iparraguirre 2021).

**2. Identifizierte Korrelationscluster:**
Die Heatmap bestätigt empirisch die vier a-priori-Cluster aus Abschnitt 7.3.
Besonders der kardiometabolische Cluster (HighBP, HighChol, HeartDiseaseorAttack)
und der Gesundheits-/Mobilitätscluster (GenHlth, PhysHlth, DiffWalk) zeigen
erkennbare Blockstruktur. Die VIF-Werte > 5 in diesen Clustern sind ein Signal,
dass bei linearen Modellen Feature-Selektion oder Regularisierung notwendig sein wird.

**3. Definitiv ausgeschlossene Features:**
Features, die alle vier Schwellenwerte unterschreiten (voraussichtlich CholCheck,
AnyHealthcare, NoDocbcCost), tragen keine messbare Information zur
Klassentrennung bei. CholCheck ist strukturell konstant (~96 % positiv);
AnyHealthcare und NoDocbcCost sind von Selektionsbias überlagert (Notebook 01,
Abschnitt 4: asymmetrisches Label-Noise). Ihr Ausschluss verringert das Risiko,
dass Modelle Rauschen statt Signal lernen.

### Ausblick auf Notebook 05

Notebook 05 (Feature Construction) baut auf der hier erarbeiteten Shortlist auf.
Die Korrelationscluster motivieren drei Konstruktionsideen:
- **Composite-Score** für den kardiometabolischen Cluster (HighBP + HighChol +
  HeartDiseaseorAttack als Summen-Feature)
- **Hurdle-Encoding** für MentHlth und PhysHlth (bereits in Notebook 03
  vorbereitet)
- **Interaktionsterme** zwischen GenHlth und Age (klinisch plausibel: der
  Alterseffekt auf Diabetesrisiko ist im schlechten Gesundheitszustand stärker)

Ob diese Konstruktionen die CV-Performance erhöhen, wird empirisch in Notebook 05
evaluiert — keine Annahmen vorab.